# Data Preprocessing & Validation — Module 1 Foundations

This notebook covers essential data preprocessing techniques for ML pipelines:
- Data loading and exploration
- Missing value handling
- Feature encoding (categorical, numerical)
- Data validation with Great Expectations
- Train/test split with stratification
- Pipeline construction with scikit-learn

**Learning Outcomes:**
1. Build reproducible preprocessing pipelines
2. Implement data quality checks
3. Handle common data issues systematically
4. Create versioned, validated datasets

In [ ]:
# Imports and Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Data validation (install with: pip install great-expectations)
try:
    import great_expectations as gx
    GE_AVAILABLE = True
except ImportError:
    GE_AVAILABLE = False
    print("Great Expectations not installed. Run: pip install great-expectations")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_style("whitegrid")
np.random.seed(42)

In [ ]:
# Load Sample Dataset — UCI Adult Income (classification)
# https://archive.ics.uci.edu/ml/datasets/Adult
from sklearn.datasets import fetch_openml

print("Fetching Adult Income dataset...")
adult = fetch_openml(data_id=1590, as_frame=True, parser='pandas')
df = adult.frame.copy()
print(f"Shape: {df.shape}")
print(f"Target: {adult.target_names}")
df.head()

In [ ]:
# Data Exploration
print("=== Basic Info ===")
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
print(f"\n=== Column Types ===")
print(df.dtypes.value_counts())
print(f"\n=== Missing Values ===")
print(df.isnull().sum().sort_values(ascending=False))
print(f"\n=== Target Distribution ===")
print(df['class'].value_counts(normalize=True))
print(f"\n=== Numerical Summary ===")
print(df.select_dtypes(include=[np.number]).describe())
print(f"\n=== Categorical Summary ===")
print(df.select_dtypes(include=['object', 'category']).describe())

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Target distribution
df['class'].value_counts().plot(kind='bar', ax=axes[0,0], title='Target Distribution')
axes[0,0].tick_params(axis='x', rotation=0)

# Age distribution by class
for cls in df['class'].unique():
    subset = df[df['class'] == cls]['age']
    axes[0,1].hist(subset, alpha=0.5, label=cls, bins=30)
axes[0,1].set_title('Age Distribution by Class')
axes[0,1].legend()

# Hours per week
df['hours-per-week'].hist(bins=30, ax=axes[0,2], edgecolor='black')
axes[0,2].set_title('Hours per Week')

# Categorical: workclass
df['workclass'].value_counts().plot(kind='barh', ax=axes[1,0], title='Workclass')
# Education
df['education'].value_counts().plot(kind='barh', ax=axes[1,1], title='Education')
# Correlation heatmap (numerical only)
num_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[num_cols].corr(), annot=True, fmt='.2f', cmap='RdBu', center=0, ax=axes[1,2])
axes[1,2].set_title('Numerical Feature Correlations')

plt.tight_layout()
plt.show()

In [ ]:
# Build Preprocessing Pipeline
# Identify feature types
target_col = 'class'
feature_cols = [c for c in df.columns if c != target_col]

numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

# Numeric pipeline: impute + scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: impute + one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'
)

# Full pipeline with classifier
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])

print("\nPipeline structure:")
print(model_pipeline)

In [ ]:
# Train/Test Split with Stratification
X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class dist: {y_train.value_counts(normalize=True).to_dict()}")
print(f"Test class dist:  {y_test.value_counts(normalize=True).to_dict()}")

# Train the pipeline
print("\nTraining model...")
model_pipeline.fit(X_train, y_train)
print("Training complete!")

In [ ]:
# Model Evaluation
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("\n=== Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=model_pipeline.classes_, 
            yticklabels=model_pipeline.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Feature importance (from Random Forest)
# Note: Need to get feature names after one-hot encoding
preprocessor_fitted = model_pipeline.named_steps['preprocessor']
cat_encoder = preprocessor_fitted.named_transformers_['cat'].named_steps['encoder']
cat_feature_names = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)

rf = model_pipeline.named_steps['classifier']
importances = pd.Series(rf.feature_importances_, index=all_feature_names).sort_values(ascending=False)

print("\n=== Top 15 Feature Importances ===")
print(importances.head(15))

# Plot top features
plt.figure(figsize=(8, 6))
importances.head(15).iloc[::-1].plot(kind='barh')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Data Validation with Great Expectations
if GE_AVAILABLE:
    # Create a context (in-memory for demo)
    context = gx.get_context(mode="ephemeral")
    
    # Create a dataframe datasource
    datasource = context.data_sources.add_pandas("pandas_datasource")
    data_asset = datasource.add_dataframe_asset(name="adult_income")
    
    # Build batch request
    batch_request = data_asset.build_batch_request(dataframe=df)
    
    # Create expectation suite
    suite = context.suites.add(gx.ExpectationSuite(name="adult_income_suite"))
    
    # Add expectations
    suite.add_expectation(gx.expectations.ExpectTableRowCountToBeBetween(min_value=30000, max_value=50000))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="age"))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="age", min_value=17, max_value=90))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(column="class", value_set=["<=50K", ">50K"]))
    suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="workclass"))
    
    # Validate
    validator = context.get_validator(batch_request=batch_request, expectation_suite=suite)
    results = validator.validate()
    
    print(f"Validation Success: {results.success}")
    print(f"\nResults Summary:")
    for result in results.results:
        status = "✓" if result.success else "✗"
        print(f"  {status} {result.expectation_config.expectation_type}")
else:
    print("Great Expectations not available. Install with: pip install great-expectations")
    print("\nManual validation checks:")
    print(f"  Row count: {len(df)} (expected 30k-50k)")
    print(f"  Age range: {df['age'].min()} - {df['age'].max()}")
    print(f"  Class values: {df['class'].unique()}")
    print(f"  Missing workclass: {df['workclass'].isnull().sum()}")

In [ ]:
# Save Pipeline and Summary
import joblib
from datetime import datetime

# Create artifacts directory
artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(exist_ok=True)

# Save the trained pipeline
model_path = artifacts_dir / "adult_income_pipeline.joblib"
joblib.dump(model_pipeline, model_path)
print(f"Pipeline saved to: {model_path}")

# Save preprocessing metadata
metadata = {
    "created_at": datetime.now().isoformat(),
    "dataset": "UCI Adult Income (OpenML ID: 1590)",
    "target": target_col,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "train_shape": X_train.shape,
    "test_shape": X_test.shape,
    "model_type": "RandomForestClassifier",
    "model_params": {"n_estimators": 100, "random_state": 42}
}

import json
meta_path = artifacts_dir / "pipeline_metadata.json"
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata saved to: {meta_path}")

# Summary
print("\n=== EXERCISE COMPLETE ===")
print("✓ Data loaded and explored")
print("✓ Preprocessing pipeline built (ColumnTransformer + Pipeline)")
print("✓ Model trained with stratified split")
print("✓ Evaluation completed (classification report, confusion matrix, feature importance)")
print("✓ Data validation with Great Expectations (or manual checks)")
print("✓ Pipeline and metadata saved for reproducibility")
print("\nNext steps:")
print("  - Experiment with different imputation strategies")
print("  - Try other classifiers (XGBoost, Logistic Regression, etc.)")
print("  - Add cross-validation for robust estimates")
print("  - Implement hyperparameter tuning")
print("  - Version data with DVC")